# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates loading and exploring the FAIR^2 dataset using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library. All dataset components are referenced by their `@id` according to the Croissant schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets and their fields and IDs.

Below, we list all record sets (using their `@id`s) and show the fields (columns) for each one.

In [ ]:
# List all record sets and fields by their @id
record_sets = dataset.record_sets
print(f"Found {len(record_sets)} record sets.")
record_set_ids = []
for rs in record_sets:
    print(f"- Record Set: {rs['@id']}")
    record_set_ids.append(rs['@id'])
    print(f"  Name: {rs.get('name', '(no name)')}")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]  # In case there is only one field
    if fields:
        print(f"  Fields/columns in this record set:")
        for field in fields:
            if isinstance(field, dict):
                print(f"    - @id: {field['@id']} -- {field.get('name', '')}")
            elif isinstance(field, str):
                print(f"    - @id: {field}")
    else:
        print("  (No fields found)")
    print('')

## 3. Data Extraction

Below, we load the data from each record set (by `@id`) into a pandas DataFrame.

Each `record_set` is referenced by its Croissant `@id` (see previous listing). We extract all rows into a DataFrame and display columns for the first record set.

In [ ]:
# Extract data from each record set
# Collect all record_set @ids
dataframes = {}
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded data for record set {record_set_id} with shape {df.shape}")
    except Exception as e:
        print(f"Could not load record set {record_set_id}: {e}")
        continue

# Use the first available record set for inspection
if dataframes:
    first_rs = list(dataframes.keys())[0]
    print(f"\nColumns for first record set ({first_rs}):")
    print(dataframes[first_rs].columns.tolist())
    dataframes[first_rs].head()
else:
    print("No record set data loaded.")

## 4. Exploratory Data Analysis (EDA)

Now select a numeric field from one of the record sets to demonstrate common EDA tasks: filtering, normalization, and aggregation.

All field names below should be referenced by their `@id`, as listed above.

In [ ]:
# Example: choose the first record set and a numeric field (@id) from its columns
if dataframes:
    df = dataframes[first_rs]
    # Try to find a numeric column
    numeric_candidates = df.select_dtypes(include=[np.number]).columns.tolist()
    print(f"Numeric fields detected: {numeric_candidates}")
    if numeric_candidates:
        numeric_field = numeric_candidates[0]  # pick the first numeric field by @id
        print(f"Using numeric field (by @id): {numeric_field}")
        threshold = df[numeric_field].quantile(0.75)  # filter above 75% percentile
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        print(filtered_df.head())

        norm_field = f"{numeric_field}_normalized"
        filtered_df[norm_field] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records (z-score):")
        print(filtered_df[[numeric_field, norm_field]].head())

        # Try groupby on a non-numeric field if available
        group_field_candidates = df.select_dtypes(exclude=[np.number]).columns.tolist()
        group_field = None
        for col in group_field_candidates:
            if col != numeric_field:
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
            print(f"Grouped data by {group_field} (mean of {numeric_field}):")
            print(grouped_df.head())
    else:
        print("No numeric fields found for EDA.")
else:
    print("No data loaded, skipping EDA.")

## 5. Visualization

Below, let's create histograms and scatterplots using the fields referenced by `@id`.

We use the first (numeric) field for histogram, and if possible, a second for scatterplot.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_candidates:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Histogram of {numeric_field} (@id)")
    plt.xlabel(numeric_field)
    plt.show()

    # If possible, scatterplot two numeric fields
    if len(numeric_candidates) >= 2:
        plt.figure(figsize=(6,6))
        sns.scatterplot(data=df, x=numeric_candidates[0], y=numeric_candidates[1])
        plt.title(f"Scatterplot: {numeric_candidates[0]} vs {numeric_candidates[1]}")
        plt.xlabel(numeric_candidates[0])
        plt.ylabel(numeric_candidates[1])
        plt.show()
else:
    print("No suitable numeric data for visualization.")

## 6. Conclusion

In this notebook, we demonstrated how to load and explore a dataset defined by a Croissant schema using the `mlcroissant` library, referencing all components by Croissant `@id`. We loaded the dataset from its Croissant URL, listed all record sets and fields by `@id`, loaded data dynamically into DataFrames, and completed a simple EDA and visualization pipeline.

This approach ensures fully FAIR and reproducible access to record sets, columns, and fields, providing transparent and scalable analysis in accordance with the Croissant Data Packaging standard.